In [9]:
import opendss_to_sincal
# import Smart_Grids.run_qsts as run_qsts
from pathlib import Path
import os 
import importlib
import run_qsts





In [10]:
OUTPUT_DIRECTORY = r"C:\Users\Matt\Documents\python_uni_projects\Smart_Grids\networks"

NETWORK = r"C:\Users\Matt\Documents\python_uni_projects\Smart_Grids\networks\LV2_43bus_PV100\LV2_43bus_PV100.sin"
PROFILES = r"C:\Users\Matt\Documents\python_uni_projects\Smart_Grids\profiles\SuRF_solar"   # folder of CSVs


### Build Network

In [3]:

# OUTPUT_DIRECTORY = r"C:\Users\Matt\Documents\python_uni_projects\Smart_Grids\networks"

sincal_projects = [
    r"C:\Users\Matt\Documents\python_uni_projects\archive\Renewable Energy\MV-associated-LV\data\LV\LV2_43bus\Master.dss",
    r"C:\Users\Matt\Documents\python_uni_projects\archive\Renewable Energy\MV-associated-LV\data\LV\LV3_55bus\Master.dss",
    r"C:\Users\Matt\Documents\python_uni_projects\archive\Renewable Energy\MV-associated-LV\data\LV\LV4_36bus\Master.dss",
]

# --- PV settings ------------------------------------------------------------
PV_PCT = 100          # % of customer buses that get PV (0 = no PV)
PV_SEED = 1          # fixed seed -> same buses every run (None = different each run)

pv = opendss_to_sincal.PVConfig(
    pct=PV_PCT,
    seed=PV_SEED,
    sizes_kva=(3.0, 5.0, 6.6),   # inverter sizes drawn at random per site
    kwp_ratio=1.0,               # DC array kWp / inverter kVA
    output_pu=1.0,               # fP: 1.0 = full output (worst case for voltage rise)
    all_lv_buses=False,          # False = only buses that carry a load
    region="A",                  # AS/NZS 4777.2 region settings
)

for project in sincal_projects:
    output_name = os.path.basename(os.path.dirname(project))
    # Tag the name so PV and no-PV versions don't overwrite each other
    if PV_PCT > 0:
        output_name += f"_PV{PV_PCT}"

    output_path = os.path.join(OUTPUT_DIRECTORY, output_name)
    os.makedirs(output_path, exist_ok=True)

    opendss_to_sincal.run_conversion(
        project, output_path, output_name,
        no_graphics=True,
        pv=pv,
    )

=== 1. Parsing OpenDSS master: C:\Users\Matt\Documents\python_uni_projects\archive\Renewable Energy\MV-associated-LV\data\LV\LV2_43bus\Master.dss
  parsing Master.dss
  parsing linecodes.dss
  parsing lines.dss
  parsing loads.dss
  parsing transformers.dss
  parsing switches.dss
  parsing groundings.dss
    circuit      : lv2_43bus
    source bus   : B3198
    base kV      : 11.0
    base freq    : 50.0 Hz
    linecodes    : 22
    lines        : 40
    switches     : 2
    transformers : 1
    loads        : 7
    reactors     : 8
  [note] linecode 'generic/lv': phase impedances differ by up to 12 % - SINCAL sequence data uses the mean
  [note] linecode 'moon_lv_oh_4wire': phase impedances differ by up to 8 % - SINCAL sequence data uses the mean
  [note] linecode 'ughv_400al_triplex_ug_4w_bundled': phase impedances differ by up to 12 % - SINCAL sequence data uses the mean
  [note] linecode 'uglv_240al_xlpe/nyl/pvc_ug_4w_bundled': phase impedances differ by up to 13 % - SINCAL sequenc

### Run Network

In [ ]:
importlib.reload(run_qsts)   # picks up edits to the script without restarting the kernel

# NETWORK = r"C:\Users\Matt\Documents\python_uni_projects\Smart_Grids\networks\LV2_43bus_PV40\LV2_43bus_PV40.sin"
# PROFILES = r"C:\Users\Matt\Documents\python_uni_projects\Smart_Grids\profiles"   # folder of CSVs

cfg = run_qsts.QSTSConfig(
    network=NETWORK,
    profiles_dir=PROFILES,
    start="2023-06-10 00:00",   # first interval
    n_intervals=288,            # 288 x 5 min = 24 h
    seed=2,                     # same seed = same profile-to-load assignment
    load_pf=0.95,
    v_max=253, v_min=216,
)

In [12]:
results = run_qsts.run_qsts(cfg)

=== 1. Network: C:\Users\Matt\Documents\python_uni_projects\Smart_Grids\networks\LV2_43bus_PV100\LV2_43bus_PV100.sin
    loads 7, PV elements 7, lines 42, transformers 1, nodes 44
    PV element  : VarShuntElement
    load phases  : L1=3, L2=2, L3=2

=== 2. Profiles: C:\Users\Matt\Documents\python_uni_projects\Smart_Grids\profiles\SuRF_solar
    29 CSVs, 29 with solar, interval 5 min, unit kW, demand = demand


ValueError: requested window 2023-05-09 00:00:00 -> 2023-05-09 23:55:00 does not overlap the profile data (2023-06-09 11:35:00 -> 2023-08-23 23:55:00).
  Pick start between 2023-06-09 11:35:00 and 2023-08-23 00:00:00,
  or pass start=None to use the intersection automatically.

Test 1

In [7]:
import sqlite3
db = r"C:\Users\Matt\Documents\python_uni_projects\Smart_Grids\networks\LV4_36bus_PV100\LV4_36bus_PV100_files\database.db"
con = sqlite3.connect(f"file:{db}?mode=ro", uri=True)

eid = con.execute(
    'SELECT Element_ID FROM "Element" WHERE Name = "Ld2827_LOAD_A"'
).fetchone()
print("element_id:", eid)

if eid:
    print("\nElement row:")
    cur = con.execute('SELECT * FROM "Element" WHERE Element_ID = ?', (eid[0],))
    print("  cols:", [d[0] for d in cur.description])
    print("  vals:", cur.fetchone())

    print("\nLoad row (if any):")
    cur = con.execute('SELECT * FROM "Load" WHERE Element_ID = ?', (eid[0],))
    row = cur.fetchone()
    if row is None:
        print("  *** NO ROW in Load table for this element ***")
    else:
        cols = [d[0] for d in cur.description]
        for c, v in zip(cols, row):
            print(f"  {c} = {v}")
con.close()

element_id: (38,)

Element row:
  cols: ['Element_ID', 'Variant_ID', 'Flag_Variant', 'Type', 'Flag_Input', 'Flag_Calc', 'Flag_Private', 'Flag_State', 'Name', 'ShortName', 'Description', 'VoltLevel_ID', 'Group_ID', 'Zone_ID', 'EcoStation_ID', 'EcoField_ID', 'Ti', 'Ts', 'TextVal', 'ci', 'Cs', 'cm', 'coo', 'Tl', 'Theta_i', 'Theta_u', 'fCe', 'fAkt', 'Metered', 'Report_No']
  vals: (38, 1, 1, 'Load', 1, 0, 0, 1, 'Ld2827_LOAD_A', 'Ld2827_L', '', 3, 0, 0, 0, 0, None, None, None, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0, 1)

Load row (if any):
  Element_ID = 38
  Variant_ID = 1
  Flag_Variant = 1
  Flag_Typified = 3
  Flag_Load = 1
  Flag_LoadType = 2
  Flag_Lf = 1
  P = 1.0
  Q = 0.3286841051788632
  u = 100.0
  Ul = 0.0
  Flag_Voltage = 1
  cosphi = 1.0
  S = 0.0
  I = 0.0
  Eapcon = 0.0
  Erpcon = 0.0
  P12 = 0.0
  Q12 = 0.0
  P23 = 0.0
  Q23 = 0.0
  P31 = 0.0
  Q31 = 0.0
  P1 = 0.0
  Q1 = 0.0
  P2 = 0.0
  Q2 = 0.0
  P3 = 0.0
  Q3 = 0.0
  E = 0.0
  t = 0.0
  Eap = 0.0
  Erp = 0.0
  EP

In [8]:
import sqlite3, win32com.client

db = r"C:\Users\Matt\Documents\python_uni_projects\Smart_Grids\networks\LV4_36bus_PV100\LV4_36bus_PV100_files\database.db"
sin = r"C:\Users\Matt\Documents\python_uni_projects\Smart_Grids\networks\LV4_36bus_PV100\LV4_36bus_PV100.sin"

# Force 50 kW, unit factors, in service
con = sqlite3.connect(db)
cur = con.execute('SELECT Element_ID FROM "Element" WHERE Name="Ld2827_LOAD_A"')
eid = cur.fetchone()
assert eid, "load element not found"
eid = eid[0]
con.execute(
    'UPDATE "Load" SET P=0.05, Q=0.01, fP=1.0, fQ=1.0 '
    'WHERE Element_ID=?', (eid,))
con.execute(
    'UPDATE "Element" SET Flag_State=1, Flag_Input=1 '
    'WHERE Element_ID=?', (eid,))
# Null out the PV so we can see pure load
con.execute('UPDATE "VarShuntElement" SET Plf=0, Qlf=0')
con.commit()
con.close()

# Run LF_USYM, nothing else touched
conn = f"TYP=NET;MODE=SQLITE;FILE={db};USR=Admin;PWD=;SINFILE={sin};"
srv = win32com.client.Dispatch("Sincal.SimulationSrv.28")
sim = srv.GetSimulation()
sim.Database(conn)
sim.SetParameter("IDENTIFICATION", "Name")
sim.LoadDB("LF_USYM")
sim.Start("LF_USYM")
print("Status:", sim.StatusID, "|", sim.StatusText)
if sim.StatusID != 1101:
    msgs = sim.Messages
    for i in range(1, msgs.Count + 1):
        m = msgs.Item(i)
        if m.Type in (3, 4):
            print(f"  [{m.Type}] {m.MessageId}: {m.Text}")

# Read voltages
con = sqlite3.connect(f"file:{db}?mode=ro", uri=True)
print("\nNode voltages:")
for r in con.execute(
        'SELECT n."Name", r."U1" FROM "ULFNodeResult" r '
        'JOIN "Node" n ON n."Node_ID" = r."Node_ID" '
        'WHERE n."Name" IN ("B1609", "B1420")'):
    print(f"  {r[0]}: U1 = {r[1]*1000:.2f} V")

print("\nTransformer flow (Tx377):")
for r in con.execute(
        'SELECT r."P", r."Q" FROM "ULFBranchResult" r '
        'JOIN "Terminal" t ON t."Terminal_ID" = r."Terminal1_ID" '
        'JOIN "Element" e ON e."Element_ID" = t."Element_ID" '
        'WHERE e."Name" = "Tx377"'):
    print(f"  P = {r[0]*1000:.1f} W, Q = {r[1]*1000:.1f} var")
con.close()

Status: 1102 | Simulation finished with errors
  [4] 2534: Power flow data has to be available to calculate the network model
  [4] 2534: Power flow data has to be available to calculate the network model
  [4] 2534: Power flow data has to be available to calculate the network model
  [4] 2534: Power flow data has to be available to calculate the network model
  [4] 2534: Power flow data has to be available to calculate the network model
  [4] 2534: Power flow data has to be available to calculate the network model
  [4] 2534: Power flow data has to be available to calculate the network model
  [4] 2534: Power flow data has to be available to calculate the network model
  [4] 2534: Power flow data has to be available to calculate the network model
  [4] 2534: Power flow data has to be available to calculate the network model
  [4] 2534: Power flow data has to be available to calculate the network model
  [4] 2534: Power flow data has to be available to calculate the network model
  [4]